# Linear Regression Health Costs Calculator

This notebook builds a neural-network regression model to predict individual healthcare expenses from demographic and lifestyle information.

**Project workflow:** load the dataset → encode categorical variables → split into 80% training and 20% testing data → train a regression model → evaluate Mean Absolute Error (MAE).

The freeCodeCamp challenge requires the test MAE to be below **$3,500**. Run the official final test cell from the original freeCodeCamp notebook as well before submitting.


## 1. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)


## 2. Load the healthcare costs dataset

In [ ]:
# The original freeCodeCamp notebook supplies this dataset in its first cells.
# This URL is a public copy of the standard insurance.csv dataset.
DATA_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

if not os.path.exists("insurance.csv"):
    df = pd.read_csv(DATA_URL)
else:
    df = pd.read_csv("insurance.csv")

df.head()


## 3. Inspect and prepare the data

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
df.describe(include="all")


In [ ]:
# Convert categorical columns (sex, smoker, region) into numeric indicator columns.
dataset = pd.get_dummies(df, columns=["sex", "smoker", "region"], dtype=float)

# Separate the target from the features.
labels = dataset.pop("expenses").astype("float32")
features = dataset.astype("float32")

# Split into 80% training data and 20% testing data.
train_dataset, test_dataset, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(train_dataset))
print("Testing rows:", len(test_dataset))
print("Feature columns:", train_dataset.columns.tolist())


## 4. Build the regression model

In [ ]:
# Normalize input features using statistics learned only from the training data.
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(np.asarray(train_dataset))

model = tf.keras.Sequential([
    tf.keras.Input(shape=(train_dataset.shape[1],)),
    normalizer,
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)

model.summary()


## 5. Train the model

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_mean_absolute_error",
    patience=30,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    train_labels,
    validation_split=0.20,
    epochs=500,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


## 6. Evaluate on the unseen test dataset

In [ ]:
test_loss, test_mae = model.evaluate(test_dataset, test_labels, verbose=2)

print(f"Test Mean Absolute Error: ${test_mae:,.2f}")
if test_mae < 3500:
    print("Success: MAE is below $3,500.")
else:
    print("The model has not reached the required MAE yet. Try additional training or model tuning.")


## 7. Plot training history

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["mean_absolute_error"], label="Training MAE")
plt.plot(history.history["val_mean_absolute_error"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("Mean Absolute Error")
plt.title("Training and Validation MAE")
plt.legend()
plt.grid(True)
plt.show()


## 8. Compare actual and predicted expenses

In [ ]:
predictions = model.predict(test_dataset, verbose=0).flatten()

results = pd.DataFrame({
    "Actual expenses": np.asarray(test_labels),
    "Predicted expenses": predictions
})
results["Absolute error"] = abs(results["Actual expenses"] - results["Predicted expenses"])
results.head(10)


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(test_labels, predictions, alpha=0.6)
lower = min(float(np.min(test_labels)), float(np.min(predictions)))
upper = max(float(np.max(test_labels)), float(np.max(predictions)))
plt.plot([lower, upper], [lower, upper], "r--")
plt.xlabel("Actual expenses ($)")
plt.ylabel("Predicted expenses ($)")
plt.title("Actual vs Predicted Healthcare Costs")
plt.grid(True)
plt.show()


## 9. Save the trained model (optional)

The saved model can be loaded later with `tf.keras.models.load_model(...)`.


In [ ]:
model.save("health_costs_regression_model.keras")
print("Model saved.")
